# Explorar el modelo de mundo de un run (Colab, desde un link de Drive)

Abre una página donde mueves al agente y ves, lado a lado, **la realidad**, lo que el modelo **entiende** al ver (reconstrucción) y lo que **sueña** sin ver, más la recompensa, el valor y la política que asigna. El run se baja de **Google Drive con su link**: no hace falta montar Drive ni agregar un acceso directo (basta que tu cuenta tenga acceso a la carpeta). Solo se baja lo necesario (`latest.pt` y la configuración, ~130 MB); los eventos de TensorBoard y el buffer, que pesan gigas, se dejan.

Sirve un entorno de **CPU**. Pasos: 1. cambia las variables de la celda 1; 2. corre las celdas en orden. La primera vez, instalar tarda unos minutos.

**Importante:** `RAMA` tiene que ser una rama de GitHub que **sepa construir el ambiente del run** (por ejemplo, que entienda `env.obstacles`) y que incluya `explorar_modelo.py` y `drive_run.py`. Si no, la celda del paso 5 lo dirá.

## 1. Qué abrir

In [ ]:
RAMA = "pruebas_server"  # rama del repo con el código (subida a GitHub)
REPO_URL = "https://github.com/FabriRandon/r2dreamer-cookie.git"

# Link (o id) de la carpeta de Drive: un run, o una carpeta que contiene runs.
CARPETA_DRIVE = "https://drive.google.com/drive/folders/11bZVL6RSUcWfm5tLzD28foWVQgtRHtMa"
NOMBRE_RUN = None  # si la carpeta tiene varios runs, escribe el nombre, por ejemplo "distinct_s0"

import os
os.environ["RAMA"], os.environ["REPO_URL"] = RAMA, REPO_URL

## 2. Dar acceso a tu Drive

Colab pide permiso una vez. Usa la cuenta que tiene acceso a la carpeta del run.

In [ ]:
from google.colab import auth

auth.authenticate_user()

## 3. Instalar el código

Clona la rama y arma un entorno aparte con Python 3.11 (el que pide el repo), con Gradio para la página. Hay que repetirlo en cada sesión nueva.

In [ ]:
%%bash
set -e
if [ -d /content/r2dreamer-cookie ]; then
  cd /content/r2dreamer-cookie && git fetch origin "$RAMA" && git checkout "$RAMA" && git pull --ff-only origin "$RAMA"
else
  git clone -b "$RAMA" "$REPO_URL" /content/r2dreamer-cookie
fi
pip install -q uv
cd /content/r2dreamer-cookie
[ -d .venv ] || uv venv --python 3.11 .venv
uv pip install --python .venv/bin/python -e ".[cookie]" gradio matplotlib 2>&1 | tail -2

## 4. Bajar el run de Drive

Solo `latest.pt`, `.hydra/`, `status.json` y `metrics.jsonl`. Si ya estaban en `/content/run`, no los vuelve a bajar.

In [ ]:
import importlib, sys

sys.path.insert(0, "/content/r2dreamer-cookie")
import drive_run

importlib.reload(drive_run)
RUN = str(drive_run.descargar(CARPETA_DRIVE, "/content/run", nombre=NOMBRE_RUN))
print("Run en", RUN)

## 5. Comprobar que carga

Arma el ambiente del run, carga el agente y hace un recorrido corto sin página. Si falla, el mensaje dice por qué (por ejemplo, que el código no conoce los obstáculos de ese run).

In [ ]:
!cd /content/r2dreamer-cookie && .venv/bin/python explorar_modelo.py --logdir "{RUN}" --prueba /content/prueba.png --device cpu 2>&1 | tail -n 12

from pathlib import Path
from IPython.display import Image as Img, display

if Path("/content/prueba.png").exists():
    print("De izquierda a derecha: vista real | reconstrucción (viendo) | sueño (12 pasos sin ver) | error")
    display(Img("/content/prueba.png"), Img("/content/prueba.actor.png"))

## 6. Cómo le fue al run (métricas)

Lo que quedó en `metrics.jsonl`: galletas del actor en evaluación y en entrenamiento, botonazos, casillas visitadas y, si el run tiene obstáculos, los choques contra ellos.

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

filas = []
for linea in open(f"{RUN}/metrics.jsonl"):
    try:
        filas.append(json.loads(linea))
    except json.JSONDecodeError:
        pass
df = pd.DataFrame(filas)
GRAFICOS = [("episode/eval_score", "galletas del actor (evaluación)"), ("episode/score", "galletas en entrenamiento"),
            ("episode/presses", "veces que aprieta el botón"), ("episode/cells", "casillas distintas visitadas"),
            ("episode/bumps", "choques contra obstáculos"), ("train/loss/image", "pérdida de reconstrucción")]
GRAFICOS = [g for g in GRAFICOS if g[0] in df]
fig, ejes = plt.subplots(2, 3, figsize=(14, 6), sharex=True)
for eje, (clave, titulo) in zip(ejes.flat, GRAFICOS):
    serie = df[["step", clave]].dropna().set_index("step")[clave]
    eje.plot(serie.index, serie.rolling(20, min_periods=1).mean() if len(serie) > 60 else serie, marker="o" if len(serie) < 40 else None)
    eje.set_title(titulo, fontsize=10)
    eje.grid(alpha=0.3)
plt.tight_layout()
plt.show()
estado = Path(RUN, "status.json")
print(estado.read_text() if estado.exists() else "sin status.json")

## 7. Abrir el explorador

Se abre aquí mismo, en un marco. Si prefieres una pestaña aparte, descomenta la última línea de la celda. Cada run corre en su propio proceso y puerto.

In [ ]:
import socket, subprocess, time, urllib.request
from google.colab import output

REPO = "/content/r2dreamer-cookie"


def abrir_explorador(run, alto=1500):
    # Cierra solo el explorador anterior de ESTE run.
    subprocess.run(["pkill", "-9", "-f", f"explorar_modelo.py --logdir {run} --port"])
    time.sleep(2)
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        puerto = s.getsockname()[1]
    archivo_log = "/content/explorar.log"
    proceso = subprocess.Popen(
        [f"{REPO}/.venv/bin/python", "-u", "explorar_modelo.py", "--logdir", str(run), "--port", str(puerto), "--device", "cpu"],
        cwd=REPO, stdout=open(archivo_log, "w"), stderr=subprocess.STDOUT,
    )
    for _ in range(300):
        if proceso.poll() is not None:
            raise RuntimeError(open(archivo_log).read()[-3000:])
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{puerto}", timeout=1)
            break
        except Exception:
            time.sleep(1)
    print(f"Run: {run}  (puerto {puerto})")
    output.serve_kernel_port_as_iframe(puerto, height=alto)
    # Para abrirlo en una pestaña aparte: output.serve_kernel_port_as_window(puerto)


abrir_explorador(RUN)

### Cómo usarlo

- **← → ↑** (o los botones): izquierda, derecha y avanzar. Haz clic una vez dentro de la página para que tome el teclado.
- Cada acción mueve el entorno real **y** el sueño. *Viendo* (posterior) recibe la imagen real en cada paso; *soñando* (prior) solo recibe las acciones desde la última sincronización.
- **S** sincroniza el sueño con la realidad. **A** juega la acción más probable del actor.
- Arriba a la izquierda está el **mapa completo** (lo iluminado es lo que ve el agente): ahí ves dónde están los obstáculos y la galleta.

### Qué mirar en un run con obstáculos

| Prueba | Qué te dice |
|---|---|
| Camina hasta quedar de frente a un obstáculo y avanza varias veces | En la realidad no te mueves. ¿El sueño también se queda quieto, o sigue avanzando como si fuera piso? Si avanza, el modelo no aprendió que bloquea |
| Mira un obstáculo, gira para dejar de verlo y vuelve | ¿La reconstrucción lo recuerda, o lo borra al perderlo de vista? |
| Desmarca *Mover también el entorno real* y sueña alrededor de uno | Qué cree el modelo que hay alrededor de él |
| Aprieta el botón (avanzando contra él) | ¿Aparece la galleta en el sueño y sube la *recompensa predicha*? Si el run casi nunca la vio, no |
| Mira el error del sueño | Sube cuando el modelo alucina: cerca de los obstáculos o en lugares que nunca visitó |